# 06｜Give Me Some Credit：Kaggle Baseline

**可執行版本｜Pandas → LogisticRegression → ROC-AUC → submission.csv**

本 Notebook 由課堂提供的 `06_GiveMeSomeCredit_baseline.py` 拆解而來，**核心模型與資料處理設定維持一致**。可在 **Google Colab** 或 VS Code 的 Jupyter Notebook 執行。依序執行每一個 Code cell，即可在取得正式資料的前提下產生 Kaggle 提交檔案。

### 執行前先做

1. 前往 [Give Me Some Credit／Data](https://www.kaggle.com/competitions/GiveMeSomeCredit/data)，依平台規定取得 `cs-training.csv`、`cs-test.csv`、`sampleEntry.csv`。**本教材不附競賽原始資料。**
2. **Colab**：在左側「檔案」上傳這三個 CSV；**VS Code**：把三個 CSV 放在 Notebook 的目前工作目錄，或自行修改檔案路徑。
3. 執行下一格確認檔案後，再往下訓練模型。若檔案尚未上傳，不要直接執行讀取 CSV 的格子。
4. 下載或提交 `submission.csv` 前，先檢查欄位、順序與預測機率。**實際線上分數必須由 Kaggle 回傳，本 Notebook 不會自動產生排行榜分數。**

> **未成年參賽者提醒：**學生若不符合 Kaggle 帳號或競賽資格，請由教師示範線上提交；學生可在本機完成程式、ROC-AUC 驗證與結果比較。

## 0｜確認資料檔案

三個 CSV 必須存在目前執行目錄。沒有檔案時會列出缺少的名稱；請先上傳，不要把真實競賽資料填入程式碼。

In [ ]:
from pathlib import Path

files = ["cs-training.csv", "cs-test.csv", "sampleEntry.csv"]
missing = [name for name in files if not Path(name).is_file()]
if missing:
    print("請先上傳：", ", ".join(missing))
else:
    print("三個 CSV 都已準備完成，可以繼續執行。")

## 1｜匯入套件、讀取正式 CSV

`train` 用於學習與本地驗證；`test` 是 Kaggle 需要預測的資料；`sample` 是官方提交格式範本。`cs-test.csv` **沒有可用的真實答案**，不能拿來直接計算 ROC-AUC。

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

train = pd.read_csv("cs-training.csv")
test = pd.read_csv("cs-test.csv")
sample = pd.read_csv("sampleEntry.csv")
print("訓練資料：", train.shape)
print("競賽測試資料：", test.shape)
print("提交範本：", sample.shape)

## 2｜建立特徵 X 與標籤 y

`SeriousDlqin2yrs` 是本次要預測的 0／1 標籤；`Unnamed: 0` 是原始資料的索引欄，不當作模型特徵。使用 `X.columns` 從測試資料挑選**相同名稱、相同順序**的欄位，避免欄位錯位。

In [ ]:
target = "SeriousDlqin2yrs"
X = train.drop(columns=["Unnamed: 0", target], errors="ignore")
y = train[target]
X_submit = test[X.columns]

print("特徵數：", X.shape[1])
print("標籤分布：")
print(y.value_counts())
print("缺失值前幾名：")
print(X.isna().sum().sort_values(ascending=False).head())

## 3｜切分訓練集與驗證集

使用 80% 的資料訓練，20% 作本地驗證。`stratify=y` 盡量維持 0、1 兩類比例；`random_state=42` 方便重現相同切分。**不要使用驗證答案訓練模型。**

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("訓練：", len(X_train), "筆")
print("驗證：", len(X_val), "筆")

## 4｜建立 Pipeline 並訓練 LogisticRegression

這裡保留 `.py` 的三個步驟：

- `SimpleImputer(strategy="median")`：以**訓練資料的中位數**補缺失值。
- `StandardScaler()`：將特徵調整到適合模型的尺度。
- `LogisticRegression(max_iter=1000)`：學習分類機率。

放在 Pipeline 裡，可確保驗證集使用**從訓練集學到**的補值與縮放設定，減少資料洩漏。

In [ ]:
model = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    LogisticRegression(max_iter=1000)
)
model.fit(X_train, y_train)
print("模型訓練完成。")

## 5｜取得風險機率，計算本地 ROC-AUC

`predict_proba(X_val)[:, 1]` 取出預測為**類別 1** 的機率（0～1），而不是最終 0／1 分類答案。Kaggle 此競賽使用 **ROC-AUC**，因此驗證也使用相同的指標。此分數是**本地驗證分數**，不等於 Kaggle 榜單分數。

In [ ]:
val_prob = model.predict_proba(X_val)[:, 1]
auc = roc_auc_score(y_val, val_prob)
print("Validation ROC-AUC:", round(auc, 5))

## 6｜用全部訓練資料重新訓練，再預測競賽測試集

確定 baseline 的驗證方式之後，依原始 `.py` 重新以全部正式訓練資料 `X, y` 訓練模型，讓提交模型能使用較多資料學習。**一旦做了這步，不應再將相同模型對先前驗證集的分數當成獨立驗證結果。**

In [ ]:
model.fit(X, y)
test_prob = model.predict_proba(X_submit)[:, 1]
print("待提交的預測：", len(test_prob), "筆")

## 7｜檢查欄位，輸出 submission.csv

依據 `sampleEntry.csv` 的欄位與列順序填入預測機率：`Id`、`Probability`。**不要輸出 `SeriousDlqin2yrs`，也不要把預測機率改成 0／1。**

In [ ]:
assert len(test_prob) == len(sample), "測試筆數與提交範本不同"
assert list(sample.columns) == ["Id", "Probability"], "提交欄位名稱不正確"
assert (test_prob >= 0).all() and (test_prob <= 1).all(), "存在超出 0～1 的機率"

sample["Probability"] = test_prob
sample.to_csv("submission.csv", index=False)
print("已建立 submission.csv：", len(sample), "筆")
print(sample.head())

## 8｜再次讀取，確認檔案無誤

上傳 Kaggle 前，檢查檔案的欄位、筆數、缺失值與預測機率範圍。

In [ ]:
submission = pd.read_csv("submission.csv")
assert list(submission.columns) == ["Id", "Probability"]
assert len(submission) == len(sample)
assert submission["Probability"].notna().all()
assert submission["Probability"].between(0, 1).all()
print(submission.head())
print("submission.csv 檢查通過。")

## 9｜下載檔案、上傳 Kaggle、記錄成績

**Google Colab 下載：**以下程式只會在 Colab 中呼叫下載；VS Code 可以直接從目前資料夾找到 `submission.csv`。

In [ ]:
try:
    from google.colab import files
    files.download("submission.csv")
except ImportError:
    print("非 Colab 環境：請在目前資料夾取得 submission.csv。")

1. 到 [競賽頁面](https://www.kaggle.com/competitions/GiveMeSomeCredit)，以**符合資格的帳號**確認能否進行 Late Submission。
2. 選擇提交入口，上傳 `submission.csv`；實際介面與提交資格以 Kaggle 登入後顯示為準。
3. 將 Kaggle 顯示的 **Score／ROC-AUC** 記錄下來，與本地驗證分數及歷史排行榜比較。**不要把不同資料集上的分數當成完全等價。**
4. 想改進時，一次只調整一個設定，例如 `max_iter`、補值方式或新增模型；**保持本地驗證切分不變**，再比較 v1、v2。

### AI 輔助修改程式的 Prompt

> 我是高一學生，正在做 Kaggle Give Me Some Credit 的 Python 專題。請協助修改目前使用 Pandas、SimpleImputer、StandardScaler、LogisticRegression 的程式。請保留 `random_state=42`、`stratify=y` 與 ROC-AUC 驗證方式，一次只改一個設定，告訴我修改的理由、可能問題與如何重新匯出 `Id,Probability` 的 submission.csv。不要假設已經取得 Kaggle 榜單成績。

**提醒：**信用風險模型涉及公平性、隱私與可能的偏差；競賽分數只是學習指標，不代表適合直接用來決定真實個人的信用資格。